# AllYourBase: CDSE optical fallback validation

Run the existing CLI for the three approved 4 km access-test windows and
2026-W34 through W41. This reads one eligible acquisition per area, with seven
native-grid optical bands. It does not run activity analysis or radar Process calls.

Use a CPU runtime. Full processing, interrupted-backfill resume and shared
writer locking remain future work. The source ZIP is pinned by its SHA-256;
the installed dependencies come from its `uv.lock`.

Before running:
1. Create a key pair in the [CDSE S3 key manager](https://eodata-s3keysmanager.dataspace.copernicus.eu/).
   Save the secret when displayed; CDSE shows it only once.
2. Add `CDSE_S3_ACCESS_KEY_ID` and `CDSE_S3_SECRET_ACCESS_KEY` in Colab's
   **Secrets** sidebar and enable notebook access. Do not put values in cells.
3. Have `allyourbase-colab-source.zip` ready to upload in the first code cell.

[S3 documentation](https://documentation.dataspace.copernicus.eu/APIs/S3.html)
| [Colab Secrets API](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py)


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

SOURCE_SHA256 = "687f96a2112500c2c82b1a1eeaaa694eab59acef4321dacdfc90027aeb88b23b"
uploaded = files.upload()
if len(uploaded) != 1 or "allyourbase-colab-source.zip" not in uploaded:
    raise RuntimeError("Upload only allyourbase-colab-source.zip.")
source_bytes = uploaded.pop("allyourbase-colab-source.zip")
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:
    raise RuntimeError("Source ZIP hash mismatch; use the ZIP delivered with this notebook.")
source_zip = Path("/content/allyourbase-colab-source.zip")
source_zip.write_bytes(source_bytes)
del source_bytes, uploaded
project = Path("/content") / ("allyourbase-src-" + SOURCE_SHA256[:12])
project.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for member in archive.infolist():
        target = (project / member.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise RuntimeError("Invalid source ZIP path.")
        if target.exists() and target.read_bytes() != archive.read(member):
            raise RuntimeError("Existing source differs; use a fresh Colab runtime.")
    archive.extractall(project)
manifest = json.loads((project / "source-manifest.json").read_text())
for name, digest in manifest["files"].items():
    if hashlib.sha256((project / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError("Extracted source hash mismatch.")
print("Verified source bundle:", SOURCE_SHA256)
print("Base revision:", manifest["base_revision"])

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.7"], check=True)
subprocess.run(
    ["uv", "sync", "--locked", "--extra", "access", "--no-dev", "--python", "3.12"],
    cwd=project,
    check=True,
)

workspace = Path("/content/allyourbase-data")
runtime_directory = Path("/content/allyourbase-runtime")
runtime_directory.mkdir(exist_ok=True)
settings = json.loads(
    subprocess.check_output(
        [
            "uv",
            "run",
            "--no-sync",
            "python",
            "-c",
            "import json; from pathlib import Path; from allyourbase.loading import load_data; "
            "print(json.dumps(load_data(Path('config/settings.yaml'))))",
        ],
        cwd=project,
        text=True,
    )
)
settings["runtime"].update(profile="colab", workspace=str(workspace))
runtime_settings = runtime_directory / "settings.json"
runtime_settings.write_text(json.dumps(settings, indent=2))
cli = [
    "uv",
    "run",
    "--no-sync",
    "allyourbase",
    "--root",
    str(project),
    "--settings",
    str(runtime_settings),
]
print("Runtime: Colab CPU; working data:", workspace)

## Check configuration and preview the bounded request

The two S3 secrets are read into subprocess memory only. OAuth is separate and
is not needed for this optical-only run. `configured` means present, not yet
verified by S3. The next retrieval is the authentication and pixel-access check.


In [ ]:
from google.colab import userdata


def optical_environment():
    environment = os.environ.copy()
    environment["ALLYOURBASE_HOME"] = str(workspace)
    try:
        for name in ("CDSE_S3_ACCESS_KEY_ID", "CDSE_S3_SECRET_ACCESS_KEY"):
            value = userdata.get(name)
            if not value:
                raise ValueError("empty secret")
            environment[name] = value
    except Exception:
        raise RuntimeError(
            "Add both S3 keys in Colab Secrets and enable notebook access."
        ) from None
    return environment


def check_readiness():
    environment = optical_environment()
    result = subprocess.run(
        cli + ["providers", "doctor"],
        cwd=project,
        env=environment,
        capture_output=True,
        text=True,
        check=True,
    )
    row = next(row for row in json.loads(result.stdout) if row["provider"] == "cdse_stac")
    print("CDSE optical pixel credentials:", row["pixel_credentials"])
    if row["pixel_credentials"] != "configured":
        raise RuntimeError("S3 credentials are missing.")


check_readiness()
spike_args = [
    "providers",
    "spike",
    "--aoi-file",
    "docs/proposals/phase-1-aois.geojson",
    "--provider",
    "cdse_stac",
    "--weeks",
    "8",
    "--end-week",
    "2026-W41",
    "--pixels",
]
subprocess.run(cli + spike_args + ["--plan"], cwd=project, check=True)

## Run once, then verify the saved files

This creates a new report folder and preserves previous runs. Do not rerun
retrieval merely to repeat verification. On failure, export the saved report
below so the explicit failure status can be reviewed.


In [ ]:
import shutil


def run_optical_spike():
    before = set((workspace / "spikes").glob("*"))
    environment = optical_environment()
    result = subprocess.run(cli + spike_args, cwd=project, env=environment, check=False)
    created = set((workspace / "spikes").glob("*")) - before
    if len(created) != 1:
        raise RuntimeError("Expected one new report folder; inspect the external workspace.")
    return result.returncode, created.pop()


spike_exit_code, run_directory = run_optical_spike()
shutil.copyfile(project / "source-manifest.json", run_directory / "source-manifest.json")
shutil.copyfile(runtime_settings, run_directory / "runtime-settings.json")
print("Report folder:", run_directory)
print("Access check:", "pass" if spike_exit_code == 0 else "failed; export report for review")

In [ ]:
validation_passed = False
if spike_exit_code == 0:
    result = subprocess.run(
        cli
        + [
            "providers",
            "validate-optical",
            "--report-dir",
            str(run_directory),
            "--aoi-file",
            "docs/proposals/phase-1-aois.geojson",
        ],
        cwd=project,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode == 0:
        summary = json.loads(result.stdout)
        validation_passed = summary["site_count"] == 3 and summary["file_count"] == 21
        print(
            "Pixel validation:",
            summary["status"],
            "| sites:",
            summary["site_count"],
            "| band files:",
            summary["file_count"],
        )
    else:
        print("Pixel validation failed; export the report for review.")
else:
    print("Access failed; no successful pixel validation is claimed.")

## Export evidence before disconnecting

Colab VM files are temporary. Download this ZIP or optionally save the completed
ZIP to Drive. Only this run's reports, source manifest, settings (variable names
only) and chips are archived. Credentials are not written to files.

A passing `optical-validation.json` verifies bands, acquisition provenance,
georeferencing, bounded dimensions, finite valid pixels and SHA-256 hashes.
Network byte counts remain unknown; native grids do not establish clear sky.


In [ ]:
evidence_zip = Path(
    shutil.make_archive(
        str(Path("/content") / ("allyourbase-evidence-" + run_directory.name)),
        "zip",
        root_dir=run_directory.parent,
        base_dir=run_directory.name,
    )
)
print("Evidence SHA-256:", hashlib.sha256(evidence_zip.read_bytes()).hexdigest())
print("Validated:", validation_passed)
files.download(str(evidence_zip))

In [ ]:
# Optional: save the completed unique archive, never a live database, to Drive.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/AllYourBase/access-evidence")
    destination.mkdir(parents=True, exist_ok=True)
    final_path = destination / evidence_zip.name
    if final_path.exists():
        raise RuntimeError("Archive already exists; preserve the existing evidence.")
    temporary = destination / (evidence_zip.name + ".tmp")
    shutil.copyfile(evidence_zip, temporary)
    if (
        hashlib.sha256(temporary.read_bytes()).digest()
        != hashlib.sha256(evidence_zip.read_bytes()).digest()
    ):
        raise RuntimeError("Drive archive hash mismatch; local archive is preserved.")
    temporary.replace(final_path)
    print("Saved completed archive:", final_path)